# Geometric Field Modulation for Ternary Parameter Golf

Phase 0: Measure ternary quantization error structure (DECISION GATE).
If structure exists → proceed with geometric field G experiments.
If no structure → stop and write up negative result.

In [ ]:
# === 1. SETUP ===
import shutil, os, glob

from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/parameter-golf'

OFFICIAL = '/content/parameter-golf'
AUX = '/content/parameter-golf-aux'
WORK = '/content/ternary_work'
TERNARY_SRC = f'{OFFICIAL}/records/track_10min_16mb/2026-03-24_74M_Ternary_UNet_FP8_10L_8192BPE_YaRN_NeoMuon'

# Clone repos
%cd /content
if not os.path.exists(OFFICIAL):
    !git clone --depth 1 https://github.com/openai/parameter-golf.git
else:
    %cd {OFFICIAL}
    !git pull
if not os.path.exists(AUX):
    !git clone https://github.com/jamesconde/parameter-golf-aux.git
else:
    %cd {AUX}
    !git pull

!pip install -q sentencepiece huggingface-hub datasets tqdm zstandard

# Create working directory
os.makedirs(WORK, exist_ok=True)
os.makedirs(f'{WORK}/geometric_field', exist_ok=True)
os.makedirs(f'{WORK}/data/tokenizers', exist_ok=True)

# Copy ternary training script + tokenizer
shutil.copy2(f'{TERNARY_SRC}/train_gpt_cuda_ternary.py', f'{WORK}/train_gpt_cuda_ternary.py')
shutil.copy2(f'{TERNARY_SRC}/fineweb_8192_bpe.model', f'{WORK}/fineweb_8192_bpe.model')
shutil.copy2(f'{TERNARY_SRC}/fineweb_8192_bpe.vocab', f'{WORK}/fineweb_8192_bpe.vocab')
shutil.copy2(f'{TERNARY_SRC}/fineweb_8192_bpe.model', f'{WORK}/data/tokenizers/fineweb_8192_bpe.model')

# Copy our analysis scripts
for f in glob.glob(f'{AUX}/geometric_field/*.py'):
    shutil.copy2(f, f'{WORK}/geometric_field/{os.path.basename(f)}')

print(f'Working directory: {WORK}')

In [ ]:
# === 2. DOWNLOAD SP8192 DATA (from HuggingFace) ===
%cd {WORK}

data_dir = f'{WORK}/data/datasets/fineweb10B_sp8192'

if os.path.exists(data_dir) and glob.glob(f'{data_dir}/fineweb_val_*.bin'):
    print(f'Data already exists at {data_dir}')
else:
    print('Downloading sp8192 data from HuggingFace...')
    from huggingface_hub import snapshot_download

    # Try as model repo first (the setup.sh uses 'hf download' without --repo-type)
    for repo_type in [None, "model", "dataset"]:
        try:
            rt_label = repo_type or "default"
            print(f'  Trying repo_type={rt_label}...')
            snapshot_download(
                repo_id="sproos/parameter-golf-tokenizers",
                repo_type=repo_type,
                allow_patterns=["datasets/fineweb10B_sp8192/*"],
                local_dir="./data",
            )
            break
        except Exception as e:
            print(f'  Failed: {str(e)[:100]}')
    else:
        # Fallback: use the official parameter-golf downloader to tokenize from sp1024
        print('  All HF downloads failed.')
        print('  Falling back: download sp1024 data and retokenize...')
        # Actually, we can use the official cached_challenge_fineweb.py if sp8192 is available
        %cd {OFFICIAL}
        !python3 data/cached_challenge_fineweb.py --variant sp8192 --train-shards 1 || true
        %cd {WORK}
        # Symlink if it was downloaded there
        src = f'{OFFICIAL}/data/datasets/fineweb10B_sp8192'
        if os.path.exists(src) and not os.path.exists(data_dir):
            os.makedirs(os.path.dirname(data_dir), exist_ok=True)
            os.symlink(src, data_dir)

train_files = glob.glob(f'{data_dir}/fineweb_train_*.bin')
val_files = glob.glob(f'{data_dir}/fineweb_val_*.bin')
print(f'Train shards: {len(train_files)}, Val shards: {len(val_files)}')
if not val_files:
    print('\nERROR: No data downloaded. You may need to:')
    print('1. Get a HuggingFace token: https://huggingface.co/settings/tokens')
    print('2. Run in a cell: from huggingface_hub import login; login()')
    print('3. Re-run this cell')

In [ ]:
# === 3. PATCH TERNARY SCRIPT FOR NON-HOPPER GPU ===
%cd {WORK}

!python3 geometric_field/patch_ternary.py train_gpt_cuda_ternary.py

# Verify
!python3 -c "import py_compile; py_compile.compile('train_gpt_cuda_ternary.py', doraise=True); print('Syntax OK')"

import torch
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name} ({vram_gb:.0f}GB)')

In [ ]:
# === 4. TRAIN TERNARY BASELINE (~1 hour) ===
# Patch saves raw state_dict as final_model_raw_sd.pt before quantization
%cd {WORK}

import os, shutil

if os.path.exists('final_model_raw_sd.pt'):
    print(f'Raw state_dict exists ({os.path.getsize("final_model_raw_sd.pt")/1e6:.1f} MB)')
elif os.path.exists(f'{DRIVE_DIR}/ternary/final_model_raw_sd.pt'):
    shutil.copy2(f'{DRIVE_DIR}/ternary/final_model_raw_sd.pt', 'final_model_raw_sd.pt')
    print(f'Loaded from Drive ({os.path.getsize("final_model_raw_sd.pt")/1e6:.1f} MB)')
else:
    print('Training ternary baseline (~1 hour)...')
    !RUN_ID=ternary_baseline \
     DATA_PATH=./data/datasets/fineweb10B_sp8192 \
     TOKENIZER_PATH=./data/tokenizers/fineweb_8192_bpe.model \
     VOCAB_SIZE=8192 \
     NUM_LAYERS=10 MODEL_DIM=768 NUM_HEADS=8 NUM_KV_HEADS=4 MLP_MULT=4 \
     EMBED_DIM=254 \
     BITNET_GROUP_SIZE=128 \
     ACTIVATION=relu2 \
     ROPE_TYPE=yarn YARN_MAX_LEN=2048 ROPE_BASE=5000 \
     LOGIT_SOFTCAP=10 SOFTCAP_TYPE=poly \
     QK_GAIN_INIT=2.25 \
     MATRIX_OPTIMIZER=muon MUON_BACKEND_STEPS=3 \
     MUON_MOMENTUM=0.95 MUON_MOMENTUM_WARMUP_START=0.85 MUON_MOMENTUM_WARMUP_STEPS=500 \
     MUON_WD=0.0 ADAM_WD=0.05 ADAM_LR=0.05 \
     MATRIX_LR=0.04 SCALAR_LR=0.02 TIED_EMBED_LR=0.02 HEAD_LR=0.02 \
     WARMDOWN_FRACTION=0.2 \
     FP_STORAGE=FP8 \
     TRAIN_BATCH_TOKENS=524288 TRAIN_SEQ_LEN=1024 \
     BIGRAM_HASH=0 SMEAR=0 DIFF_ATTN=0 MLP_GROUPS=0 MTP_HEADS=0 \
     TIE_EMBEDDINGS=1 \
     ITERATIONS=10000 MAX_WALLCLOCK_SECONDS=3600 \
     WARMUP_STEPS=5 VAL_LOSS_EVERY=500 TRAIN_LOG_EVERY=100 \
     USE_COMPILE=0 \
     SEED=42 \
     python3 train_gpt_cuda_ternary.py

In [ ]:
# === 5. SAVE CHECKPOINTS TO DRIVE ===
import shutil, os
os.makedirs(f'{DRIVE_DIR}/ternary', exist_ok=True)

for f in ['final_model_raw_sd.pt', 'final_model.ternary.ptz']:
    if os.path.exists(f):
        shutil.copy2(f, f'{DRIVE_DIR}/ternary/{f}')
        print(f'Saved {f} ({os.path.getsize(f)/1e6:.1f} MB) → Drive')
    else:
        print(f'{f} not found')

In [ ]:
# === 6. PHASE 0: QUANTIZATION ERROR STRUCTURE ANALYSIS (DECISION GATE) ===
%cd {WORK}

!python3 geometric_field/phase0_analysis.py \
    --checkpoint final_model_raw_sd.pt \
    --model-dim 768 --num-heads 8 --num-kv-heads 4 --mlp-mult 4 --num-layers 10 \
    --group-size 128 \
    --output geometric_field/phase0_results.json

In [ ]:
# === 7. SAVE RESULTS + DISPLAY CLASSIFICATION ===
import shutil, json, os

for f in ['geometric_field/phase0_results.json', 'geometric_field/phase0_results_profiles.pt']:
    if os.path.exists(f):
        os.makedirs(f'{DRIVE_DIR}/ternary/results', exist_ok=True)
        shutil.copy2(f, f'{DRIVE_DIR}/ternary/results/{os.path.basename(f)}')

if os.path.exists('geometric_field/phase0_results.json'):
    with open('geometric_field/phase0_results.json') as f:
        results = json.load(f)
    c = results['classification']
    print('=' * 50)
    print(f"Scenario: {c['scenario']}")
    print(f"PROCEED: {c['proceed']}")
    print(f"Row structure:  {c['mean_row_structure']:.2f} (threshold: 2.0)")
    print(f"Col structure:  {c['mean_col_structure']:.2f} (threshold: 2.0)")
    print(f"Layer variation: {c['layer_variation']:.3f} (threshold: 0.15)")
    print('=' * 50)
    if c['proceed']:
        print('\n-> Column structure detected. Proceed with G experiments.')
    else:
        print('\n-> No exploitable structure. Write up as negative result.')
else:
    print('Phase 0 results not found. Run the analysis cell first.')